# 04. 점포 월 매출 특성 분석: 선형 회귀

- **분석 배경**
  - 기술통계와 가설검정 실습에서는 점포 유형이나 지역처럼 변수 하나씩 매출을 비교했습니다.
  - 월 매출에는 여러 변수가 함께 영향을 주므로 다른 조건을 고정했을 때 각 특성이 매출을 얼마나 바꾸는지 알고자 합니다.
- **분석 대상**
  - 100개 점포의 2024년 1월 ~ 2025년 12월 점포 × 월 매출입니다.
  - 평상시 월 매출과 관련된 특성을 찾는 것이 목적이므로, 평상시와 다른 달이 있으면 원인을 확인한 뒤 제외합니다.
- **분석 질문**
  - 점포의 월 매출은 어떤 특성(영업일수, 행사, 날씨, 면적, 지역 등)에 따라 달라지는가?
- **사용 데이터**
  - `store_month_clean.pkl`(기술통계 실습에서 결합·보정한 점포 × 월 2,400행)
- **분석 단위**: 점포 × 월

**학습 목표**

1. 범주형 변수를 더미 변수로 바꿔 선형 회귀 모형을 적합하고, 회귀계수와 결정계수를 해석할 수 있다.
2. 요약 정보의 잔차 표, 잔차 그래프 및 Breusch-Pagan 검정으로 선형 회귀 모형의 오차 가정을 진단할 수 있다.
3. 매출과 같은 비율형 목표변수를 로그로 변환하고, 로그 선형 회귀계수를 % 변화로 해석할 수 있다.
4. 다중공선성과 영향점을 진단하고 대응할 수 있다.
5. 단순 비교와 '다른 조건을 통제한' 회귀 결과가 왜 다를 수 있는지 설명할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import statsmodels.api as sm
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

기술통계 실습에서 점포 마스터를 결합하고 매출 입력 오류를 보정해 저장한 pkl 파일을 불러옵니다.<br>
회귀분석은 극단값 하나에도 회귀계수가 크게 흔들리므로, 입력 오류를 보정한 데이터를 사용합니다.<br>
pkl 파일이 없으면 기술통계 실습 노트북을 먼저 실행합니다.

In [ ]:
# 기술통계 실습에서 저장한 pkl 파일을 읽고 df에 할당합니다.
df = pd.read_pickle(filepath_or_buffer='../data/store_month_clean.pkl')

# df의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
df

In [ ]:
# df의 정보를 확인합니다.
df.info()

## 1. 상관분석

**질문**: 월 매출과 각 입력변수의 상관계수는 얼마이고, 관계는 어떤 모양인가?

In [ ]:
# 월별 입력변수 이름을 리스트로 생성합니다.
month_cols = ['open_days', 'weekend_days', 'holiday_days', 'holiday_season', 'event_days', 'rain_days', 'extreme_days']

# 점포 규모 입력변수 이름을 리스트로 생성합니다.
size_cols = ['floor_area', 'sku_count', 'parking']

# 입력변수별 매출과의 Pearson 상관분석을 한 번에 실행합니다.
# [참고] columns=[리스트, 리스트]는 첫 번째 리스트의 각 변수를 두 번째 리스트의 변수와 짝지어 검정합니다.
corr_table = pg.pairwise_corr(data=df, columns=[month_cols + size_cols, ['sales']], method='pearson')

# Pearson 상관분석 결과를 상관계수 기준으로 오름차순 정렬하고 소수점 4자리까지 반올림하여 확인합니다.
corr_table.sort_values(by='r').round(decimals=4)

In [ ]:
# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 매장 면적과 월 매출의 산점도를 그립니다.
sns.scatterplot(data=df, x='floor_area', y='sales', alpha=0.3, ax=axes[0])
axes[0].set(title='매장 면적과 월 매출', xlabel='매장 면적 (m²)', ylabel='월 매출 (백만 원)')

# 영업일수와 월 매출의 산점도를 그립니다.
sns.scatterplot(data=df, x='open_days', y='sales', alpha=0.3, ax=axes[1])
axes[1].set(title='영업일수와 월 매출', xlabel='영업일수', ylabel='월 매출 (백만 원)')
plt.tight_layout()
plt.show()

**해석**
- Pearson 상관분석 결과입니다.
  - 매장 면적, 품목 수, 주차면수처럼 **점포 규모**를 나타내는 변수의 상관이 가장 큽니다.
  - 주말 영업일수와 영업일수는 유의확률이 0.05보다 커서 유의하지 않았지만, 이 변수들을 입력변수 목록에서 바로 제외하면 안 됩니다.
  - 주말 영업일수와 영업일수 같은 월별 변수는 점포 간 규모 차이가 워낙 커서 월별 변화가 가려지기 때문에 상관관계가 작게 나온 것입니다.
- 단순 상관은 다른 변수의 영향을 통제하지 않은 결과이므로, 도메인 지식에 따라 여러 입력변수로 적합한 다중 선형 회귀 결과로 판단합니다.
  - 4절 로그 선형 회귀에서 매장 면적을 함께 넣으면 영업일수는 유의하게 나옵니다.
  - 관측값이 많으면 상관계수가 작아도 상관분석 결과는 유의하게 나오므로 관계의 크기는 `p_unc`가 아니라 `r`로 판단합니다.
- 산점도를 보면 매장 면적이 클수록 매출이 크고, 면적이 커질수록 점들이 위아래로 더 넓게 퍼집니다. [깔때기 모양]
  - 이 모양은 3절에서 다시 다룹니다.
- 영업일수 산점도는 뚜렷한 방향 없이 세로 띠 모양으로 흩어져 있습니다.
  - 영업일수가 대부분 26 ~ 29일로 비슷하고, 같은 영업일수 안에서도 점포 규모에 따라 매출이 크게 다르기 때문입니다.
  - 왼쪽 끝에 떨어져 있는 두 점은 영업일수가 유난히 적은 달로, 5절 영향점에서 다시 다룹니다.
- 주차면수는 상관계수가 크지만 **상관관계가 인과관계를 의미하는 것은 아닙니다.**
  - 상권이 큰 지역은 점포 면적과 매출에 동시에 영향을 미칠 수 있습니다. [교란변수]
  - 매장 면적이 클수록 주차면수도 증가하며, 주차면수와 매출의 관계는 함께 움직이는 것일 수 있습니다.
  - 점포 규모를 나타내는 변수끼리 상관이 높은 문제는 7절 다중공선성에서 다시 다룹니다.

## 2. 입력변수 행렬과 목표변수 벡터 준비

**질문**: 지역 같은 범주형 변수는 회귀식에 어떻게 들어가는가? 더미 변수로 변환합니다.

In [ ]:
# 지역을 더미 변수로 변환하고 dummies에 할당합니다.
dummies = pd.get_dummies(data=df['region'], prefix='region', drop_first=True, dtype=int)

# 원래 지역과 더미 변수를 옆으로 붙이고 dummy_view에 할당합니다.
dummy_view = pd.concat(objs=[df['region'], dummies], axis=1)

# 지역별로 한 행씩 확인합니다.
# [참고] 지역의 고유값은 4개이므로 4행이 출력됩니다.
dummy_view.drop_duplicates(subset='region').sort_values(by='region')

**해석**
- 범주가 k개이면 0/1 더미 변수를 k − 1개 만듭니다.
  - 정렬 순서상 첫 번째 범주가 기준 범주가 되고, 나머지 범주마다 더미 변수가 하나씩 생깁니다.
  - 지역에서 `수도권`이 **기준 범주**이고, 각 더미의 회귀계수는 '기준 범주와 비교한 차이'로 해석합니다.
- 점포 유형은 매장 면적을 구간으로 나눈 변수라 면적과 정보가 겹칩니다.
  - 점포 유형은 더미 변수로 만들지 않고, 연속형 변수인 면적을 입력변수로 사용합니다.

**질문**: 회귀 모형에 넣을 입력변수 행렬을 만듭니다.<br>
월별 변수, 점포 규모 변수(면적·품목 수·주차면수), 지역 더미 변수를 모두 사용합니다.

In [ ]:
# 지역 더미 변수 이름을 리스트로 생성합니다.
region_cols = ['region_영남권', 'region_충청·강원권', 'region_호남권']

# 월별 변수, 점포 규모 변수, 지역 더미 변수로 원 척도 입력변수 행렬을 생성합니다.
X_raw = pd.concat(objs=[df[month_cols + size_cols], dummies[region_cols]], axis=1)

# 원 척도 입력변수 행렬 X_raw의 처음 5행을 확인합니다.
X_raw.head()

**해석**
- 한 행이 점포 × 월이고, 열이 입력변수입니다.
- 점포 규모 변수(면적·품목 수·주차면수)는 서로 상관이 높지만, 일단 모두 넣어 적합하고 7절 다중공선성에서 점검합니다.

## 3. 원 척도 선형 회귀 모형 적합

**질문**: 매출을 그대로 목표변수로 쓰는 선형 회귀 모형을 적합합니다.

In [ ]:
# 원 척도 선형 회귀 모형을 적합합니다.
# [참고] hds.stat.ols()는 입력변수 행렬에 `const`(상수항)을 자동으로 추가합니다.
raw_model = hds.stat.ols(y=df['sales'], X=X_raw)

# 원 척도 선형 회귀 모형의 요약 정보를 확인합니다.
raw_model.summary()

**해석**
- 요약 정보는 다음 순서로 읽습니다.
  - ① 모형의 유의성(`Prob (F-statistic)`) → ② 회귀계수의 유의성(`P>|t|`) → ③ 결정계수(`R-squared`, `Adj. R-squared`) → ④ 잔차의 정규성, 독립성 및 자기상관(맨 아래 표)
- `F-statistic`은 '모든 회귀계수가 0이다'라는 귀무가설을 검정하는 F-검정의 통계량입니다.
  - `Prob (F-statistic)`이 0.05보다 작으므로 귀무가설을 기각하고, 입력변수 중 적어도 하나는 매출과 관련이 있다고 판단합니다.
- `coef`(회귀계수)는 다른 입력변수를 고정한 상태에서 해당 변수가 1 증가하면 매출(백만 원)이 얼마나 변하는지를 뜻합니다.
  - `P>|t|`가 0.05보다 작으면 해당 변수의 회귀계수가 0이 아니라고 판단합니다.
- `R-squared`(결정계수)는 매출의 변동 중 모형이 설명하는 비율입니다.
  - `Adj. R-squared`(조정된 결정계수)는 변수를 많이 넣을수록 R²가 저절로 커지는 것을 보정한 값입니다.
  - 다만 표본이 크면 쓸모없는 변수가 있어도 두 값의 차이가 작으므로, 이것만으로 변수 선택을 판단하지 않습니다.
  - 가상 데이터라 현실에서 다루는 데이터보다 설명력이 높은 편입니다.
  - 실제 데이터에서는 점포 고유의 특성(상권, 경쟁점 등)에 의해 결정계수가 훨씬 낮게 나옵니다.

- 맨 아래 표는 잔차에 대한 정보입니다.

  | 항목 | 뜻 | 이번 결과 |
  |---|---|---|
  | `Omnibus`<br>`Jarque-Bera(JB)` | '잔차가 정규분포를 따른다'라는 정규성 검정 | `Prob(Omnibus)`와 `Prob(JB)`가 0.05보다 작아<br> 잔차가 정규분포를 따른다고 보기 어려움 |
  | `Skew`(왜도) | 좌우 대칭이면 0 | 0.248로 대칭에 가까움 |
  | `Kurtosis`(첨도) | 정규분포이면 3 | 5.317로 3보다 커서 정규분포보다 큰 잔차가 자주 나옴 |
  | `Durbin-Watson` | 이웃한 행의 잔차끼리 **자기상관** 여부 확인(0~4)<br>2에 가까우면 자기상관이 없음 | 1.489로 2보다 작음(양의 자기상관) |

  [참고] 두 정규성 검정은 잔차의 왜도와 첨도로 계산합니다.

- 정규성 가정은 회귀계수를 추정하는 데는 필요하지 않고, 표본이 작을 때 t-검정·F-검정과 신뢰구간을 정확하게 계산하는 데 필요합니다.
  - 표본이 크면 중심극한정리에 따라 회귀계수의 분포가 정규분포에 가까워지므로, 잔차가 정규분포를 따르지 않아도 유의확률과 신뢰구간을 근사적으로 쓸 수 있습니다.
  - 또 표본이 크면 정규성 검정은 아주 작은 차이도 기각하므로, 유의확률보다 왜도·첨도의 크기와 `Normal Q-Q` 플롯을 함께 봅니다.
  - 현재 데이터는 2,400행이므로 정규성보다 다음 단계에서 확인할 잔차 분산(등분산성)이 더 중요한 문제입니다.
- `Durbin-Watson`이 2보다 작은 것은 데이터가 점포별로 월 순서로 정렬되어 있어 이웃한 행이 같은 점포의 연속된 달이기 때문입니다.
  - 같은 점포의 잔차끼리 비슷하다는 뜻이며, 6절 군집 표준오차에서 다룹니다.
- 결정계수가 높아 보이지만 회귀계수와 유의확률을 믿으려면 먼저 **잔차가 회귀분석의 가정을 만족하는지** 확인해야 합니다.

**질문**: 잔차 그래프에서 잔차의 퍼짐이 일정한가?

In [ ]:
# 잔차 그래프를 그립니다.
hds.stat.regression_diagnosis(model=raw_model)
plt.show()

**해석**
- 잔차 그래프 4개는 다음과 같이 읽습니다.

  | 그래프 | 위치 | 확인하는 것 | 가정을 만족하는 모양 |
  |---|---|---|---|
  | `Residuals vs Fitted` | 왼쪽 위 | 선형성 | 잔차가 0을 중심으로 고르게 퍼짐 |
  | `Normal Q-Q` | 오른쪽 위 | 정규성 | 거의 모든 점들이 대각선 위에 놓임 |
  | `Scale-Location` | 왼쪽 아래 | 등분산성 | 빨간 선이 수평에 가까움 |
  | `Influence Plot` | 오른쪽 아래 | 영향점 | 다른 점들과 동떨어진 점이 없음(5절에서 다룸) |

- `Residuals vs Fitted`에서 추정값이 클수록 잔차가 위아래로 넓게 퍼지는 **깔때기 모양**을 보입니다.
  - 깔때기 모양이란 추정값이 증가함에 따라 분산이 점차 커지는 모습을 의미합니다.
  - 빨간 선도 0에서 벗어나 휘어 있어, 직선 관계를 가정한 원 척도 모형이 매출과 면적의 '곱해지는' 관계를 잘 맞추지 못한다는 신호입니다. (4절에서 다룸)
- `Scale-Location`의 빨간 선도 오른쪽으로 갈수록 올라갑니다.
  - 잔차의 분산이 일정하지 않은 **이분산**입니다.
- `Normal Q-Q`의 왼쪽은 아래로, 오른쪽은 위로 말리는 모습은 정규분포보다 꼬리가 두꺼워 큰 잔차가 자주 나온다는 뜻입니다.
  - 특히 오른쪽 꼬리가 긴 분포를 보이고 있으므로 로그 변환을 고려할 필요가 있습니다.
- `Influence Plot`으로 회귀 모형에 큰 영향을 미치는 관측값이 있는지 확인합니다.
  - 점의 크기가 클수록 영향점으로 의심합니다.

**질문**: Breusch-Pagan 검정으로 이분산을 확인합니다.<br>
(귀무가설: 잔차의 분산이 일정하다)

In [ ]:
# 잔차의 등분산성 가정을 확인합니다.
hds.stat.breusch_pagan(model=raw_model)

**해석**
- `P-Value`가 0.05보다 작으므로 '잔차의 분산이 일정하다'는 가설을 기각합니다.
  - 잔차 그래프에서 본 잔차의 이분산이 Breusch-Pagan 검정으로도 확인되었습니다.
- 이분산이 있으면 회귀계수 자체는 크게 틀리지 않아도 **표준오차와 유의확률을 믿기 어렵습니다.**
  - 매출에 로그를 취해 이분산 문제를 해결해 봅니다.

## 4. 로그 선형 회귀 모형 적합

**질문**: 면적이 2배씩 커질 때 매출은 같은 금액만큼 늘어나는가, 같은 배수만큼 늘어나는가?

In [ ]:
# 면적 구간 경계를 최소 면적(3,300㎡)부터 2배씩 커지도록 생성합니다.
area_bins = [3300, 6600, 13200, 26400]

# 면적 구간별 점포 수, 평균 면적, 평균 매출을 계산하고 area_table에 할당합니다.
area_table = df.groupby(by=pd.cut(x=df['floor_area'], bins=area_bins, right=False), observed=True) \
    .agg(stores=('store_id', 'nunique'), floor_area=('floor_area', 'mean'), sales=('sales', 'mean'))

# 앞 구간 대비 평균 면적의 배수를 계산해서 평균 면적 오른쪽에 추가합니다.
area_table.insert(loc=2, column='area_ratio', value=area_table['floor_area'] / area_table['floor_area'].shift(periods=1))

# 앞 구간 대비 평균 매출의 차이를 계산합니다.
area_table['diff'] = area_table['sales'].diff()

# 앞 구간 대비 평균 매출의 배수를 계산합니다.
area_table['sales_ratio'] = area_table['sales'] / area_table['sales'].shift(periods=1)

# area_table을 소수점 3자리까지 반올림해서 확인합니다.
area_table.round(decimals=3)

**해석**
- 면적 구간을 2배 간격으로 나눴더니, 앞 구간 대비 평균 면적은 1.731배, 1.872배로 커지고 평균 매출은 1.632배, 1.638배로 커집니다.
  - 면적이 비슷한 배수로 커질 때 매출도 거의 같은 배수만큼 커집니다.
- 반면 평균 매출의 차이는 1,512백만 원에서 2,493백만 원으로 커집니다.
  - 매출이 면적에 따라 같은 금액씩 '더해지는' 것이 아니라 같은 배수씩 '곱해지는' 관계라는 뜻입니다.
- 원 척도 회귀계수 하나('면적 1㎡당 매출 ○백만 원')로는 이 관계를 요약할 수 없습니다.
  - 면적이 큰 점포일수록 금액 차이가 커지는 것은 3절 잔차 그래프의 깔때기 모양과도 연결됩니다.
- 로그는 배수를 간격으로 바꾸는 변환입니다.
  - 매출과 면적에 모두 로그를 취하면 '면적이 같은 배수로 커질 때 매출도 같은 배수로 커지는' 관계가 직선이 됩니다.

**질문**: 목표변수를 `log(매출)`, 면적·품목 수·주차면수를 로그로 바꾼 모형을 적합합니다.<br>
앞의 표처럼 면적이 같은 배수로 커질 때 매출도 같은 배수로 커지므로, 매출과 면적에 모두 로그를 취해야 둘의 관계가 직선이 됩니다.

In [ ]:
# X_raw에서 점포 규모 변수를 제거하고 X_log에 할당합니다.
X_log = X_raw.drop(columns=size_cols)

# 점포 규모 변수에 로그를 취하고 열 이름 앞에 log_를 붙여 log_size에 할당합니다.
log_size = np.log(df[size_cols]).add_prefix(prefix='log_')

# 월별 변수, 로그 점포 규모 변수, 지역 더미 변수 순서로 X_log를 다시 생성합니다.
X_log = pd.concat(objs=[X_log[month_cols], log_size, X_log[region_cols]], axis=1)

# 매출에 로그를 취한 목표변수 벡터를 생성합니다.
y_log = np.log(df['sales'])

In [ ]:
# 로그 선형 회귀 모형을 적합합니다.
log_model = hds.stat.ols(y=y_log, X=X_log)

# 로그 선형 회귀 모형의 요약 정보를 확인합니다.
log_model.summary()

**해석**
- 로그 선형 회귀계수는 '변수가 1 증가하면 매출이 약 (계수 × 100)%만큼 변한다'로 읽습니다.
  - 회귀계수가 크면 9절의 (e^계수 − 1) × 100으로 계산합니다.
- `log_floor_area`처럼 로그를 취한 입력변수의 회귀계수는 **탄력성**입니다.
  - 면적이 1% 커질 때 매출이 몇 % 커지는지를 뜻합니다.
- 잔차 표에서 `Prob(Omnibus)`와 `Prob(JB)`가 0.05보다 커져서, 잔차가 정규분포를 따르지 않는다고 볼 근거가 부족합니다.
  - `Kurtosis`(첨도)가 3에 가까워졌고 `Skew`(왜도)도 0에 가깝습니다.
- `Durbin-Watson`은 여전히 2보다 작습니다.
  - 매출에 로그를 취해도 같은 점포의 달끼리 잔차가 비슷한 것은 그대로이기 때문입니다.

**질문**: 로그 선형 회귀 모형의 잔차는 퍼짐이 일정해졌는가?

In [ ]:
# 잔차 그래프를 그립니다.
hds.stat.regression_diagnosis(model=log_model)
plt.show()

In [ ]:
# 잔차의 등분산성 가정을 확인합니다.
hds.stat.breusch_pagan(model=log_model)

**해석**
- `Residuals vs Fitted`에서 깔때기 모양은 사라졌습니다.
  - 추정값이 8.0 근처에서 잔차의 절댓값이 큰 점이 몇 개 있습니다.
  - Breusch-Pagan 검정의 `P-Value`가 여전히 0.05보다 작으므로, 이 소수의 점이 원인일 수 있습니다.
- `Influence Plot`에서도 다른 점들과 동떨어진 점이 몇 개 보입니다.
  - 5절 영향점 분석에서 확인합니다.

## 5. 영향점 확인

회귀 결과를 크게 바꾸는 행을 **영향점**이라고 합니다.<br>
어떤 관측값이 영향점인지 여부는 두 가지를 함께 봅니다.

- `std_resid`(**스튜던트 잔차**): 회귀 직선에서 얼마나 멀리 떨어져 있는가?
- `hat`(**레버리지**): 입력변수의 중심에서 얼마나 멀리 떨어져 있는가?

`cooksd`(**Cook의 거리**)는 두 값을 함께 반영해, 그 행을 빼고 다시 추정하면 회귀계수가 얼마나 바뀌는지를 나타냅니다.

### 단순 선형 회귀로 영향점 이해하기

**질문**: 로그 면적 하나만 넣은 단순 선형 회귀 모형에서 스튜던트 잔차와 레버리지는 어떻게 나타나는가?

In [ ]:
# 로그 면적만 넣은 입력변수 행렬을 생성합니다.
X_simple = X_log[['log_floor_area']]

# 단순 선형 회귀 모형을 적합합니다.
simple_model = hds.stat.ols(y=y_log, X=X_simple)

# 관측값별 영향점 정보를 simple_aug에 할당합니다.
simple_aug = hds.stat.augment(model=simple_model)

# simple_aug를 cooksd 내림차순 정렬하고 처음 5행을 확인합니다.
# [참고] sales는 로그 매출, fitted는 추정값, std_resid는 스튜던트 잔차, hat은 레버리지, cooksd는 Cook의 거리입니다.
simple_aug.sort_values(by='cooksd', ascending=False).head()

In [ ]:
# Cook의 거리가 가장 큰 2행의 인덱스를 outlier_index에 할당합니다.
outlier_index = simple_aug['cooksd'].nlargest(n=2).index

# 로그 면적과 영향점 정보를 옆으로 붙이고 simple_view에 할당합니다.
simple_view = pd.concat(objs=[X_simple, simple_aug], axis=1)

# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 로그 면적과 로그 매출의 산점도와 회귀 직선을 그리고, Cook의 거리가 가장 큰 2행을 빨간색으로 표시합니다.
sns.scatterplot(data=simple_view, x='log_floor_area', y='sales', alpha=0.3, ax=axes[0])
sns.lineplot(data=simple_view, x='log_floor_area', y='fitted', color='0.2', errorbar=None, ax=axes[0])
sns.scatterplot(data=simple_view.loc[outlier_index, :], x='log_floor_area', y='sales', color='red', ax=axes[0])

# 로그 면적과 로그 매출의 평균점을 검은 삼각형으로 표시합니다.
x_mean, y_mean = simple_view['log_floor_area'].mean(), simple_view['sales'].mean()
axes[0].scatter(x=x_mean, y=y_mean, marker='^', s=150, color='black', zorder=3)
axes[0].set(title='로그 면적과 로그 매출', xlabel='로그 면적', ylabel='로그 매출')

# 로그 면적과 레버리지의 산점도를 그리고, 같은 2행을 빨간색으로 표시합니다.
sns.scatterplot(data=simple_view, x='log_floor_area', y='hat', alpha=0.3, ax=axes[1])
sns.scatterplot(data=simple_view.loc[outlier_index, :], x='log_floor_area', y='hat', color='red', ax=axes[1])

# 로그 면적의 평균을 회색 점선으로 표시합니다.
axes[1].axvline(x=simple_view['log_floor_area'].mean(), color='gray', linestyle='--')
axes[1].set(title='로그 면적과 레버리지 (단순 선형 회귀 모형)', xlabel='로그 면적', ylabel='레버리지')
plt.tight_layout()
plt.show()

In [ ]:
# 빨간 점의 점포, 연월, 영업일수, 공사 여부에 영향점 정보를 옆으로 붙여 확인합니다.
df_outlier = df.loc[outlier_index, ['store_id', 'ym', 'open_days', 'renovation']]
aug_outlier = simple_aug.loc[outlier_index, ['std_resid', 'hat', 'cooksd']]
pd.concat(objs=[df_outlier, aug_outlier], axis=1).round(decimals=4)

**해석**
- 왼쪽 그래프의 빨간 점은 검정색 회귀 직선보다 한참 아래에 있으므로 스튜던트 잔차가 큽니다.
  - 빨간 점은 점포 면적에 비해 매출이 적게 나온 것입니다.
- 회귀 직선은 항상 평균점 $(\bar{x}, \bar{y})$(왼쪽 그래프의 검은 삼각형)을 지납니다.
  - 평균점을 받침점으로 하는 시소를 연상할 수 있습니다.
  - 시소는 받침점에서 먼 곳을 누를수록 작은 힘으로도 크게 움직입니다.
  - 받침점에서 멀어질수록 회귀 직선을 크게 기울일 수 있으므로 레버리지(지렛대)라고 부릅니다.
- 오른쪽 그래프처럼 단순 선형 회귀 모형의 레버리지는 로그 면적이 평균(회색 점선)에서 멀어질수록 커집니다.
  - 레버리지는 평균에서 떨어진 거리의 제곱에 비례해 커지므로, 평균에서 가장 작고 양쪽으로 갈수록 빠르게 커지는 U자 모양이 됩니다.
  - 양 끝점은 레버리지가 크지만, 회귀 직선에 가까이 있으므로(잔차가 작으므로) 회귀계수를 크게 움직이지 않습니다.
- 영향점 여부는 스튜던트 잔차와 레버리지를 **함께 반영한 Cook의 거리**로 판단합니다.
  - 빨간 점은 단순 선형 회귀 모형에서 Cook의 거리가 가장 큰 두 행입니다.
  - 빨간 점은 회귀 직선에서 멀리 떨어져 있으므로 스튜던트 잔차는 크지만, $\bar{x}$에서 멀지 않으므로 레버리지는 작은 편입니다.
  - 이어지는 다중 선형 회귀 모형에서 같은 두 행의 Cook의 거리가 어떻게 바뀌는지 비교합니다.

### 다중 선형 회귀의 영향점

**질문**: 입력변수가 여러 개인 로그 선형 회귀 모형에서도 같은 두 행의 레버리지가 작은가?

In [ ]:
# 로그 선형 회귀 모형의 관측값별 영향점 정보를 log_aug에 할당합니다.
log_aug = hds.stat.augment(model=log_model)

# 로그 면적, 영업일수와 영향점 정보를 옆으로 붙이고 multi_view에 할당합니다.
multi_view = pd.concat(objs=[X_log[['log_floor_area', 'open_days']], log_aug], axis=1)

# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 로그 면적과 레버리지의 산점도를 그리고, 단순 선형 회귀 모형에서 표시한 2행을 빨간색으로 표시합니다.
sns.scatterplot(data=multi_view, x='log_floor_area', y='hat', alpha=0.3, ax=axes[0])
sns.scatterplot(data=multi_view.loc[outlier_index, :], x='log_floor_area', y='hat', color='red', ax=axes[0])

# 로그 면적의 평균을 회색 점선으로 표시합니다.
axes[0].axvline(x=multi_view['log_floor_area'].mean(), color='gray', linestyle='--')
axes[0].set(title='로그 면적과 레버리지 (로그 선형 회귀 모형)', xlabel='로그 면적', ylabel='레버리지')

# 영업일수와 레버리지의 산점도를 그리고, 같은 2행을 빨간색으로 표시합니다.
sns.scatterplot(data=multi_view, x='open_days', y='hat', alpha=0.3, ax=axes[1])
sns.scatterplot(data=multi_view.loc[outlier_index, :], x='open_days', y='hat', color='red', ax=axes[1])

# 영업일수의 평균을 회색 점선으로 표시합니다.
axes[1].axvline(x=multi_view['open_days'].mean(), color='gray', linestyle='--')
axes[1].set(title='영업일수와 레버리지 (로그 선형 회귀 모형)', xlabel='영업일수', ylabel='레버리지')
plt.tight_layout()
plt.show()

**해석**
- 단순 선형 회귀 모형에서 봤던 같은 두 점인데, 로그 선형 회귀 모형에서는 레버리지가 다른 점보다 훨씬 큽니다.
  - 왼쪽 그래프에서 빨간 점이 면적 평균에서 크게 멀지 않음에도 레버리지가 큰 이유는 오른쪽 그래프에 있습니다.
  - 오른쪽 그래프에서 빨간 점은 영업일수 평균에서 멀리 떨어져 있습니다.
- 단순 선형 회귀 모형에서는 입력변수가 로그 면적 하나뿐이라 로그 면적만으로 레버리지가 정해지므로 U자 곡선 하나로 그려졌습니다.
- 다중 선형 회귀 모형의 레버리지는 **모든 입력변수를 함께 놓은 공간**에서 중심과 떨어진 거리입니다.
  - 그 결과 입력변수 하나로 그린 2차원 그래프에서는 평균 근처로 보이는 점도 레버리지가 클 수 있습니다.
  - 특히 빨간 점의 레버리지가 워낙 커서 세로축 범위가 넓어진 탓에 나머지 점들은 바닥에 눌려 잘 보이지 않습니다.
- 잔차 그래프의 `Influence Plot`에서 영향점이 이상한 위치에 보이는 것도 같은 이유입니다.
  - 가로축의 레버리지가 클수록, 세로축의 스튜던트 잔차가 위아래로 멀어질수록 Cook의 거리가 커집니다.
  - 영향점은 그래프 하나로 판단하지 말고 Cook의 거리 값으로 찾습니다.

**질문**: 로그 선형 회귀 모형에서 Cook의 거리 상위 행은 어느 점포의 어느 달인가?

In [ ]:
# Cook의 거리 상위 5행의 인덱스를 top5_index에 할당합니다.
top5_index = log_aug.nlargest(n=5, columns='cooksd').index

# Cook의 거리 상위 5행의 점포, 연월, 영업일수, 공사 여부에 영향점 정보를 옆으로 붙여 확인합니다.
df_top5 = df.loc[top5_index, ['store_id', 'ym', 'open_days', 'renovation']]
aug_top5 = log_aug.loc[top5_index, ['std_resid', 'hat', 'cooksd']]
pd.concat(objs=[df_top5, aug_top5], axis=1).round(decimals=4)

**해석**
- 상위 두 행의 Cook의 거리가 다른 행보다 훨씬 크고, 단순 선형 회귀 모형에서 빨간색으로 표시한 두 행과 같습니다.
  - 두 행은 같은 점포의 연속된 두 달로 영업일수가 크게 적고, `renovation`(공사 여부)가 1인 리뉴얼 공사 기간입니다.
  - 3월 중순부터 4월 중순까지 공사로 인해 전체 매장이 휴업했고, 재개장 직후에는 일정 기간 동안 하루 매출이 평소보다 낮았습니다.
- 영향점은 입력 오류와 다릅니다.
  - **실제로 일어난 일**이므로 무조건 지우지 않고, 원인을 확인한 뒤 분석 목적에 맞는지 판단합니다.
- 이 분석의 목적은 '평상시 월 매출과 관련된 특성'을 찾는 것이므로, 공사 기간처럼 평상시가 아닌 달은 제외하고 다시 추정해 비교합니다.

[참고] 공사 기간을 제외하는 대신 다음 방법을 쓸 수도 있습니다.
- 공사 여부를 더미 변수로 추가: 나머지 회귀계수는 두 행을 제외하고 추정한 결과와 거의 같습니다.
  - 다만 더미 변수의 회귀계수는 두 행으로만 추정한 값이므로 일반적인 공사 효과로 해석하지 않습니다.
- 목표변수를 로그 일평균 매출(`log(sales / open_days)`)로 변경: 이때는 `open_days`를 입력변수에서 뺍니다.
  - 회귀계수는 월 매출이 아니라 영업일 하루 매출의 변화로 해석합니다.

**질문**: 공사 기간 두 행을 제외하고 다시 추정하면 회귀계수가 어떻게 바뀌는가?

In [ ]:
# 공사 기간인 행의 인덱스를 remove_index에 할당합니다.
remove_index = df.index[df['renovation'].eq(other=1)]

# X_log에서 인덱스가 remove_index인 행을 삭제하고 X_clean에 할당합니다.
X_clean = X_log.drop(index=remove_index)

# y_log에서 인덱스가 remove_index인 행을 삭제하고 y_clean에 할당합니다.
y_clean = y_log.drop(index=remove_index)

# 영향점을 제외한 데이터로 선형 회귀 모형을 다시 적합하고 clean_model에 할당합니다.
clean_model = hds.stat.ols(y=y_clean, X=X_clean)

# 제외 전후 회귀계수를 비교합니다.
# [참고] y절편은 모든 입력변수가 0일 때의 값이라 해석하지 않습니다.
pd.DataFrame(data={'제외 전 회귀계수': log_model.params, '제외 후 회귀계수': clean_model.params}).round(decimals=4)

**해석**
- `open_days`(영업일수) 회귀계수가 눈에 띄게 줄었습니다.
  - 평상시에는 영업일수가 26 ~ 29일 사이에서 하루씩 달라지지만, 공사로 점포 문을 장기간 닫은 날의 매출이 통째로 사라집니다.
  - 여기에 재개장 직후 줄어든 매출까지 더해지면서 '영업일수가 줄면 매출이 아주 많이 준다'는 쪽으로 회귀계수를 끌고 갔던 것입니다.
- 두 달을 제외하자 `holiday_days`, `holiday_season` 같은 달력 변수의 회귀계수도 조금씩 움직였습니다.
  - 면적·지역 같은 점포 특성 회귀계수는 거의 그대로입니다.

**질문**: 영향점을 제외한 뒤 이분산 문제도 해결되었는가?

In [ ]:
# 영향점 제외 후 모형의 등분산성 가정을 확인합니다.
hds.stat.breusch_pagan(model=clean_model)

**해석**
- 영향점을 제외한 결과 Breusch-Pagan 검정의 `P-Value`가 0.05보다 커져 '분산이 일정하다'는 가설을 기각할 수 없습니다.
  - **로그 변환 + 영향점 처리**를 통해 잔차의 분산이 다르다고 볼 근거가 부족한 모형이 되었습니다.
- 다음으로 이 모형(`clean_model`)에서 회귀계수의 유의성을 확인합니다.

**질문**: 영향점을 제외한 모형에서 어떤 입력변수가 매출과 유의한 관련이 있는가? 회귀계수의 유의성을 확인합니다.

In [ ]:
# 절편을 제외한 회귀계수별 유의확률을 소수점 4자리까지 반올림해서 확인합니다.
# [참고] 요약 정보의 P>|t| 열과 같은 값입니다.
clean_model.pvalues.drop(index='const').sort_values(ascending=False).round(decimals=4)

**해석**
- `log_sku_count`(로그 품목 수)와 `log_parking`(로그 주차면수)만 유의확률이 0.05보다 큽니다.
- 큰 점포일수록 품목도 많고 주차장도 넓은데, 왜 이 두 변수는 매출과 유의한 관련이 없게 나왔을까요?
  - 7절에서 입력변수 사이의 다중공선성으로 확인합니다.
- [참고] 이 유의확률은 같은 점포가 여러 달 반복된다는 점을 반영하지 않은 값입니다.
  - 바로 다음 6절에서 군집 표준오차로 다시 확인합니다.

## 6. 군집 표준오차

**질문**: 영향점을 제외한 모형에서도 같은 점포의 잔차끼리 비슷한가? `Durbin-Watson`으로 확인합니다.

In [ ]:
# 영향점을 제외한 모형의 잔차로 Durbin-Watson 통계량을 계산합니다.
# [참고] 데이터가 점포별로 월 순서로 정렬되어 있으므로 이웃한 행은 대부분 같은 점포의 연속된 달입니다.
dw_store = sm.stats.durbin_watson(resids=clean_model.resid)

# 같은 잔차를 월, 점포 순서로 다시 정렬해서 Durbin-Watson 통계량을 계산합니다.
month_order = df.loc[X_clean.index].sort_values(by=['ym', 'store_id']).index
dw_month = sm.stats.durbin_watson(resids=clean_model.resid.loc[month_order])

# 두 정렬 순서의 Durbin-Watson 통계량을 출력합니다.
print(f'점포·월 순서: {dw_store:.3f}, 월·점포 순서: {dw_month:.3f}')

**해석**
- 영향점을 제외한 모형의 `Durbin-Watson`은 1.592로 여전히 2보다 작습니다.
  - 로그 변환과 영향점 처리로 이분산은 해결했지만, 같은 점포의 잔차끼리 비슷한 문제는 남아 있습니다.
- 같은 잔차를 월·점포 순서로 정렬하면 `Durbin-Watson`이 2.048로 2에 가까워집니다.
  - 이웃한 행이 같은 달의 다른 점포가 되므로 잔차끼리 상관이 거의 없습니다.
  - `Durbin-Watson`은 이웃한 행끼리만 비교하므로 행 순서에 따라 값이 달라집니다.
  - 따라서 2에 가깝다고 해서 같은 점포의 잔차끼리 독립이라고 판단하면 안 됩니다.
- 같은 대상이 반복되는 데이터는 행 순서와 상관없이 점포 단위로 묶어 표준오차를 다시 계산하는 **군집 표준오차**로 확인합니다.

**질문**: 같은 점포가 여러 달 반복되는 데이터에서 점포 단위 군집 표준오차를 쓰면 회귀계수와 표준오차는 어떻게 달라지는가?

In [ ]:
# 영향점을 제외한 모형에 사용한 행의 점포 ID를 clean_groups에 할당합니다.
clean_groups = df.loc[X_clean.index, 'store_id']

# 영향점을 제외한 모형을 점포 단위 군집 표준오차로 다시 추정하고 clean_cluster에 할당합니다.
clean_cluster = clean_model.model.fit(cov_type='cluster', cov_kwds={'groups': clean_groups}, use_t=True)

# 최소제곱법과 군집 표준오차의 회귀계수, 표준오차, 유의확률을 한 표로 합칩니다.
clean_compare = pd.concat(objs=[hds.stat.ols_table(model=clean_model)[['coef', 'std_err', 'p_value']],
                                hds.stat.ols_table(model=clean_cluster)[['coef', 'std_err', 'p_value']]],
                          axis=1, keys=['최소제곱법', '군집 표준오차'])

# clean_compare를 소수점 4자리까지 반올림해서 확인합니다.
clean_compare.round(decimals=4)

**해석**
- 두 모형의 회귀계수는 같고, 표준오차와 유의확률만 다릅니다.
  - 같은 점포가 24번 반복되는데도 최소제곱법(OLS)은 모든 행을 서로 독립인 관측값으로 보고 표준오차를 계산합니다.
  - 같은 점포의 24행은 비슷한 정보를 반복하므로, 실제 정보량은 2,398행보다 훨씬 적습니다.
- 면적·품목 수·주차면수·지역처럼 점포마다 하나로 정해진 변수는 군집 표준오차가 최소제곱법의 2배 이상으로 커집니다.
  - 같은 점포 안에서는 입력변수 값이 같고 잔차도 비슷하기 때문입니다.
  - 월별 변수는 주말 영업일수(약 2배)를 빼면 표준오차가 거의 같습니다.
- 주차면수의 유의확률은 0.0987에서 0.4686으로 커졌고, 품목 수는 두 방법 모두 유의하지 않습니다.
  - 나머지 변수는 군집 표준오차로 다시 계산해도 유의확률이 0.05보다 작습니다.
- 점포 × 월처럼 같은 대상을 여러 번 관측한 데이터는 군집 표준오차로 유의성을 판단합니다.
  - 8절의 부분 F-검정과 최종 모형에서도 군집 표준오차로 다시 확인합니다.

## 7. 다중공선성 확인

**질문**: `clean_model`(영향점을 제외한 모형)의 입력변수 사이에 다중공선성이 있는가? 분산팽창지수로 확인합니다.

In [ ]:
# 영향점을 제외한 모형의 입력변수별 분산팽창지수를 확인합니다.
hds.stat.vif(model=clean_model).T.sort_values(by=0, ascending=False).round(decimals=2)

**해석**
- 분산팽창지수가 10을 넘으면 다중공선성이 심하다고 봅니다.
  - `log_floor_area`(로그 면적)과 `log_sku_count`(로그 품목 수)의 분산팽창지수가 10을 넘습니다.
- `log_parking`(로그 주차면수)는 10보다 작아 다중공선성이 심하지 않지만, 월별 변수와 지역 더미 변수보다는 큽니다.

**질문**: 점포 규모 변수 3개는 얼마나 강하게 상관되어 있고, 회귀계수는 어떻게 나왔는가?

In [ ]:
# 로그 점포 규모 변수 이름을 리스트로 생성합니다.
log_size_cols = ['log_floor_area', 'log_sku_count', 'log_parking']

# 로그 점포 규모 변수끼리 Pearson 상관분석을 실행합니다.
pg.pairwise_corr(data=X_clean, columns=log_size_cols, method='pearson').round(decimals=4)

In [ ]:
# 영향점을 제외한 모형에서 점포 규모 변수의 회귀계수 검정 결과를 확인합니다.
hds.stat.ols_table(model=clean_model).loc[log_size_cols, :].round(decimals=4)

**해석**
- 세 변수의 `r`(상관계수)가 모두 높고, 특히 면적과 품목 수는 1에 매우 가깝습니다.
  - 큰 점포는 품목도 많고 주차장도 넓기 때문에, 모형이 매출 차이를 셋 중 어느 변수 덕분인지 나누지 못합니다.
- 면적이 매출 차이를 대부분 가져가고, 품목 수와 주차면수의 회귀계수는 0에 가까운 음수이며 유의하지 않습니다.
  - 품목 수의 회귀계수가 음수라고 해서 '품목 수가 많을수록 매출이 준다'로 해석하면 안 됩니다.

## 8. 최종 모형

**질문**: 분산팽창지수가 가장 큰 두 변수 중 품목 수를 빼면 분산팽창지수는 어떻게 바뀌는가?<br>
면적이 점포 규모를 가장 직접 나타내고 현업에서도 이해하기 쉬우므로 면적을 남깁니다.

In [ ]:
# X_clean에서 로그 품목 수를 제거하고 X_no_sku에 할당합니다.
X_no_sku = X_clean.drop(columns='log_sku_count')

# 로그 품목 수를 제거한 선형 회귀 모형을 적합합니다.
no_sku_model = hds.stat.ols(y=y_clean, X=X_no_sku)

# 입력변수별 분산팽창지수를 확인합니다.
hds.stat.vif(model=no_sku_model).T.sort_values(by=0, ascending=False).round(decimals=2)

In [ ]:
# 로그 품목 수를 제거한 모형을 점포 단위 군집 표준오차로 다시 추정하고 no_sku_cluster에 할당합니다.
# [참고] X_no_sku는 X_clean에서 열만 뺀 것이라 행이 같으므로 6절의 clean_groups를 그대로 사용합니다.
no_sku_cluster = no_sku_model.model.fit(cov_type='cluster', cov_kwds={'groups': clean_groups}, use_t=True)

# 점포 규모 변수의 회귀계수 검정 결과를 최소제곱법과 군집 표준오차로 비교합니다.
pd.concat(objs=[hds.stat.ols_table(model=no_sku_model)[['coef', 'std_err', 'p_value']],
                hds.stat.ols_table(model=no_sku_cluster)[['coef', 'std_err', 'p_value']]],
          axis=1, keys=['최소제곱법', '군집 표준오차']).loc[['log_floor_area', 'log_parking'], :].round(decimals=4)

**해석**
- 품목 수를 제외하면 면적의 분산팽창지수가 크게 줄었습니다.
  - 이제 모든 입력변수의 분산팽창지수가 10보다 작습니다.
- 주차면수는 분산팽창지수 기준으로는 문제가 없지만, 면적을 통제하면 여전히 유의하지 않고 회귀계수도 음수입니다.
  - 최소제곱법의 유의확률은 0.098로 0.05에 가깝지만, 군집 표준오차로 다시 계산하면 0.4694로 유의하지 않다는 것이 더 분명해집니다.
  - 주차면수는 면적과 함께 움직일 뿐이며, 매출을 따로 설명하는 몫이 없으므로 주차면수도 입력변수에서 제외합니다.

**질문**: 주차면수까지 빼면 분산팽창지수와 조정된 결정계수는 어떻게 되는가?

In [ ]:
# X_no_sku에서 로그 주차면수를 제거하고 X_final에 할당합니다.
X_final = X_no_sku.drop(columns='log_parking')

# 점포 규모 변수로 면적만 남긴 최종 모형을 적합합니다.
final_model = hds.stat.ols(y=y_clean, X=X_final)

# 입력변수별 분산팽창지수를 확인합니다.
hds.stat.vif(model=final_model).T.sort_values(by=0, ascending=False).round(decimals=2)

In [ ]:
# 세 모형의 점포 규모 변수의 회귀계수와 표준오차를 한 표로 비교합니다.
pd.DataFrame(data={'세 변수 모두: 회귀계수': clean_model.params.reindex(index=log_size_cols),
                   '세 변수 모두: 표준오차': clean_model.bse.reindex(index=log_size_cols),
                   '품목 수 제외: 회귀계수': no_sku_model.params.reindex(index=log_size_cols),
                   '품목 수 제외: 표준오차': no_sku_model.bse.reindex(index=log_size_cols),
                   '최종 모형: 회귀계수': final_model.params.reindex(index=log_size_cols),
                   '최종 모형: 표준오차': final_model.bse.reindex(index=log_size_cols)}) \
    .round(decimals=4)

In [ ]:
# 세 모형의 조정된 결정계수를 비교합니다.
print(f'세 변수 모두: {clean_model.rsquared_adj:.4f}')
print(f'품목 수 제외: {no_sku_model.rsquared_adj:.4f}')
print(f'최종 모형: {final_model.rsquared_adj:.4f}')

**해석**
- 세 변수에서 면적만 남기면 면적의 분산팽창지수가 1에 가까워집니다.
  - 분산팽창지수가 1이라는 것은 다른 입력변수들과 선형 관계가 없다는 것을 의미합니다.
- 세 변수를 모두 넣었을 때 면적 회귀계수의 표준오차가 가장 크고, 변수를 뺄수록 작아집니다.
  - 상관이 높은 변수를 함께 넣으면 회귀계수의 분산이 '팽창'해 추정이 불안정해진다는 것이 바로 이 뜻입니다.
- 두 변수를 빼도 조정된 결정계수가 그대로이므로 점포 규모 변수 중 면적만 남깁니다.

**질문**: 품목 수와 주차면수를 함께 빼도 설명력이 유의하게 줄지 않는가? 부분 F-검정과 AIC로 확인하고, 군집 표준오차로 다시 검정합니다.

In [ ]:
# 세 변수를 모두 넣은 모형과 최종 모형의 부분 F-검정을 실행합니다.
# [참고] clean_model은 완전(Full) 모형이고, final_model은 축소(Reduced) 모형입니다.
# [참고] compare_f_test()는 F-통계량, 유의확률, 자유도(뺀 입력변수의 개수)를 반환합니다.
f_stat, f_pvalue, df_diff = clean_model.compare_f_test(restricted=final_model)

# 부분 F-검정의 F-통계량, 자유도, 유의확률을 출력합니다.
print(f'F-통계량 = {f_stat:.2f}, 자유도 = {df_diff:.0f}, 유의확률 = {f_pvalue:.4f}')

In [ ]:
# 품목 수와 주차면수의 회귀계수가 모두 0이라는 가설을 6절의 군집 표준오차 모형(clean_cluster)으로 검정합니다.
# [참고] compare_f_test()는 오차의 독립을 전제로 하므로, 같은 점포가 반복되는 데이터에서는 군집 표준오차로 같은 가설을 다시 검정합니다.
print(clean_cluster.f_test(r_matrix='log_sku_count = 0, log_parking = 0'))

In [ ]:
# 두 모형의 AIC를 비교합니다.
print(f'세 변수 모두: {clean_model.aic:.2f}')
print(f'최종 모형: {final_model.aic:.2f}')

**해석**
- 요약 정보의 F-검정이 입력변수를 모두 뺀 모형과 비교한다면, 부분 F-검정은 일부 입력변수만 제외한 모형과 비교합니다.
  - 부분 F-검정의 귀무가설은 '제외한 입력변수(품목 수, 주차면수)의 모회귀계수가 모두 0이다'입니다.
- 부분 F-검정의 유의확률이 0.05보다 크므로 귀무가설을 기각하지 못합니다.
  - 두 변수를 함께 빼도 설명력이 유의하게 줄었다고 볼 수 없습니다.
- 다만 6절의 `Durbin-Watson`에서 본 것처럼 같은 점포의 잔차끼리 비슷하므로, 부분 F-검정의 전제(오차의 독립)가 맞지 않습니다.
  - 점포 단위 군집 표준오차로 같은 가설을 다시 검정해도 유의확률이 0.05보다 커서(F = 0.27, 유의확률 = 0.768) 결론은 같습니다.
  - 반복 관측 데이터에서 변수를 함께 뺄지 판단할 때는 군집 표준오차로 검정한 결과를 기준으로 합니다.
- AIC는 작을수록 좋은 모형입니다.
  - 최종 모형의 AIC가 조금 더 작습니다.
- 이후 분석은 이 모형(`final_model`)을 기준으로 합니다.

**질문**: 점포 규모 변수를 모두 넣은 모형과 최종 모형의 회귀계수는 얼마나 다른가?

In [ ]:
# 점포 규모 변수를 모두 넣은 모형과 최종 모형의 회귀계수를 한 표로 정리합니다.
coef_table = pd.DataFrame(data={'세 변수 모두: 회귀계수': clean_model.params, '최종 모형: 회귀계수': final_model.params})

# 상수항을 제외하고 입력변수 순서대로 확인합니다.
coef_table.reindex(index=clean_model.params.index).drop(index='const').round(decimals=4)

**해석**
- 품목 수와 주차면수를 빼도 나머지 입력변수의 회귀계수는 거의 바뀌지 않습니다.
  - 영향점을 제거했을 때 일부 회귀계수가 크게 바뀐 것과 다른 점입니다.

**질문**: 최종 모형도 점포 단위 군집 표준오차로 다시 추정하면 회귀계수의 유의성이 바뀌는가?

In [ ]:
# 최종 모형에 사용한 행의 점포 ID를 store_groups에 할당합니다.
store_groups = df.loc[X_final.index, 'store_id']

# 최종 모형을 점포 단위 군집 표준오차로 다시 추정합니다.
cluster_model = final_model.model.fit(cov_type='cluster', cov_kwds={'groups': store_groups}, use_t=True)

# 최소제곱법 모형의 회귀계수, 표준오차, 유의확률을 선택합니다.
ols_result = hds.stat.ols_table(model=final_model)[['coef', 'std_err', 'p_value']]

# 군집 표준오차 모형의 회귀계수, 표준오차, 유의확률을 선택합니다.
cluster_result = hds.stat.ols_table(model=cluster_model)[['coef', 'std_err', 'p_value']]

# 최소제곱법과 군집 표준오차의 결과를 한 표로 합칩니다.
cluster_compare = pd.concat(objs=[ols_result, cluster_result], axis=1, keys=['최소제곱법', '군집 표준오차'])

# cluster_compare를 소수점 4자리까지 반올림해서 확인합니다.
cluster_compare.round(decimals=4)

**해석**
- 최종 모형에서도 면적·지역처럼 점포마다 하나로 정해진 변수는 군집 표준오차가 최소제곱법의 2배 이상으로 커집니다.
- 군집 표준오차로 다시 계산해도 모든 회귀계수의 유의확률이 0.05보다 작아 결론은 바뀌지 않습니다.
- 다음 절에서는 이 모형(`final_model`)의 회귀계수를 해석합니다.

## 9. 최종 모형의 회귀계수 해석

**질문**: 최종 모형의 회귀계수를 정확한 % 변화로 바꾸면 얼마인가? (% 변화 = (e^계수 − 1) × 100)

In [ ]:
# 최종 모형의 회귀계수 검정 결과를 표로 정리합니다.
effect_table = hds.stat.ols_table(model=final_model)

# 회귀계수를 % 변화로 바꾼 열을 추가합니다.
effect_table['% 변화'] = (np.exp(effect_table['coef']) - 1) * 100

# 상수항을 제외하고 회귀계수, 유의확률, % 변화를 확인합니다.
effect_table.loc[effect_table.index != 'const', ['coef', 'p_value', '% 변화']].round(decimals=4)

**해석**
- 목표변수가 로그 변환된 경우, 다른 입력변수를 고정한 상태에서 해당 변수가 1단위 증가할 때 `% 변화`를 읽습니다.

  | 변수 | 해석 |
  |---|---|
  | `open_days` | 영업일이 1일 늘 때 월 매출이 약 2.56% 증가 (원래 매출의 102.56%) |
  | `rain_days` | 강수일이 1일 늘 때 매출이 약 0.71% 감소 (원래 매출의 99.29%) |
  | `extreme_days` | 폭염·한파일이 1일 늘 때 매출이 약 0.60% 감소 |
  | `event_days` | 행사일이 1일 늘 때 월 매출이 약 1.02% 증가 |

- 로그 선형 회귀의 효과는 더하기가 아니라 **곱하기**로 쌓입니다.
  - 행사일이 2일 늘면 매출은 $1.010196^2 \approx 1.0205$배, 즉 약 2.05% 증가합니다.
- 입력변수도 로그 변환된 경우에는 `% 변화` 열이 아니라 `coef`를 탄력성으로 읽습니다.
  - `log_floor_area`: 면적이 1% 커질 때 매출이 약 0.86% 증가합니다.

- 더미 변수는 기준 범주와의 차이로 해석합니다.

  | 변수 | 해석 |
  |---|---|
  | `holiday_season` | 설·추석이 있는 달은 다른 달보다 매출이 약 14.30% 높음 |
  | `region_영남권` | 수도권 대비 매출이 약 14.88% 낮음 (수도권의 85.12% 수준) |
  | `region_충청·강원권` | 수도권 대비 매출이 약 17.44% 낮음 (수도권의 82.56% 수준) |
  | `region_호남권` | 수도권 대비 매출이 약 24.79% 낮음 (수도권의 75.21% 수준) |

- 가설검정 실습에서 지역별 점포 매출은 Welch 분산분석 결과 유의하지 않았지만, 면적을 통제한 회귀에서는 지역 회귀계수가 모두 유의합니다.
  - 지역마다 점포 크기가 제각각이라 분산분석에서는 면적 차이에서 오는 큰 변동이 지역 차이를 가렸던 것입니다.
  - 선형 회귀에서는 '같은 크기의 점포끼리 비교하면 지역에 따라 매출이 몇 % 다르다'를 볼 수 있고, 이것이 **통제**의 효과입니다.

**질문**: 더미 변수의 효과를 그래프로 확인합니다. 수도권과 영남권의 회귀 직선은 어떤 모양인가?

In [ ]:
# 로그 면적, 로그 매출, 지역을 묶고 region_view에 할당합니다.
region_view = X_final[['log_floor_area']].assign(log_sales=y_clean, region=df['region'])

# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 수도권과 영남권의 로그 면적과 로그 매출의 산점도와 회귀 직선을 그립니다.
for region in ['수도권', '영남권']:
    sns.regplot(data=region_view.loc[region_view['region'].eq(other=region), :], x='log_floor_area', y='log_sales', 
                truncate=False, ci=None, scatter_kws={'alpha': 0.2, 's': 15}, label=region, ax=axes[0])
axes[0].set(title='수도권과 영남권의 회귀 직선', xlabel='로그 면적', ylabel='로그 매출')
axes[0].legend()

# 로그 면적의 범위로 x축을 지정합니다.
axes[1].set(xlim=(region_view['log_floor_area'].min(), region_view['log_floor_area'].max()))

# 4개 지역의 회귀 직선만 그립니다.
for region in ['수도권', '영남권', '충청·강원권', '호남권']:
    sns.regplot(data=region_view.loc[region_view['region'].eq(other=region), :], x='log_floor_area', y='log_sales', 
                truncate=False, ci=None, scatter=False, label=region, ax=axes[1])
axes[1].set(title='4개 지역의 회귀 직선', xlabel='로그 면적', ylabel='로그 매출')
axes[1].legend()
plt.tight_layout()
plt.show()

**해석**
- 왼쪽 그래프에서 두 회귀 직선의 기울기가 거의 같고, 영남권 회귀 직선은 수도권 회귀 직선을 아래로 평행 이동한 모양입니다.
  - 이 수직 간격이 위 표의 `region_영남권` 회귀계수에 해당합니다.
- 더미 변수는 기울기는 그대로 두고 절편만 바꾸는 역할을 합니다.
  - 오른쪽 그래프를 보면 4개 지역의 회귀 직선이 거의 나란하고, 수도권이 가장 위, 호남권이 가장 아래에 있습니다.
  - 영남권과 충청·강원권은 회귀 직선이 거의 겹칠 만큼 비슷한 경향을 보입니다.
- 매출 수준은 영남권이 충청·강원권보다 약 3% 높습니다.
  - [참고] 두 회귀계수의 차이를 검정하면 군집 표준오차 기준으로도 유의하지만, 이 노트북에서는 생략합니다.
  - 이 차이가 업무적으로 의미 있는 크기인지는 별도로 판단합니다.
- 오른쪽 그래프에서는 면적이 클수록 지역 간 매출 차이가 조금 줄어드는 것처럼 보이지만, 이 노트북에서는 기울기 차이를 검정하지 않았으므로 그래프만으로 판단하지 않습니다.
  - [참고] 그래프의 회귀 직선은 지역별로 로그 면적만 넣어 따로 적합한 것이라, 간격이 계수와 정확히 같지는 않습니다.

**더미 변수 회귀 모형과 범주별 회귀 모형의 비교**

| 구분 | 더미 변수 회귀 모형 | 범주별 회귀 모형 |
|---|---|---|
| 모형 수 | 전체 데이터로 1개 | 지역별로 1개씩 총 4개 |
| 가정 | 절편만 지역마다 다르고 기울기는 같음 | 절편과 기울기 모두 지역마다 다름 |
| 지역 차이 | 더미 변수의 회귀계수로 바로 비교하고 검정 | 모형이 나뉘어 있어 회귀계수 차이를 바로 검정하기 어려움 |
| 추정의 안정성 | 전체 데이터를 사용하므로 안정적 | 관측값이 적은 지역은 불안정 |
| 적합한 경우 | 지역별 기울기가 비슷할 때 | 지역별 기울기가 뚜렷이 다를 때 |

**유의사항**
- 범주별 회귀 모형의 회귀계수가 조금씩 다르더라도 표본 오차일 수 있습니다.
  - 회귀계수의 차이를 지역 차이로 해석하지 않습니다.
- 더미 변수 회귀 모형에서 지역별 회귀 직선의 기울기가 평행하지 않으면 지역 더미와 해당 입력변수의 상호작용 항을 추가합니다.
  - 특히 두 회귀 직선이 X자 모양으로 교차하는 모습이 뚜렷하면 해당 입력변수와 상호작용이 존재한다고 판단합니다.
- 각 점포마다 지역 더미가 24번씩 반복되므로, 기울기 차이를 검정할 때는 점포 단위 군집 표준오차를 사용합니다.
  - 기본 표준오차를 쓰면 표본 크기로 인해 차이가 없어도 유의하게 나오기 쉽습니다.

## 10. 최종 모형 점검

**질문**: 최종 모형(로그 선형 회귀 모형, 영향점 제외)의 잔차는 회귀분석의 가정을 만족하는가? 잔차 그래프로 확인합니다.

In [ ]:
# 최종 모형의 잔차 그래프를 그립니다.
hds.stat.regression_diagnosis(model=final_model)
plt.show()

**해석**
- 3절의 원 척도 모형과 비교하면 `Residuals vs Fitted`의 깔때기 모양과 `Scale-Location`의 오르막이 사라졌습니다.
- `Normal Q-Q`의 양 끝이 조금 벗어나는 정도만 있을 뿐, 관측값이 수천 개이므로 신뢰구간과 유의확률은 크게 영향받지 않습니다.
- `Influence Plot`에서도 혼자 크게 동떨어진 점이 없습니다.

## 정리

- 원 척도 선형 회귀는 잔차가 깔때기 모양(이분산)이었습니다.
  - 로그 변환 및 영향점 처리 후에는 잔차의 분산이 다르다고 볼 근거가 부족해졌습니다.
- 잔차의 정규성과 독립성은 요약 정보의 잔차 표로 확인합니다.
  - 로그 변환 후에는 정규성을 기각하지 못했지만, `Durbin-Watson`은 2보다 작아 같은 점포의 잔차끼리 자기상관이 있었습니다.
  - 그래서 같은 점포가 반복되는 점을 반영한 군집 표준오차로 결론이 바뀌지 않는지 확인했습니다.
- 로그 선형 회귀계수는 '다른 조건을 고정한 상태에서 1단위 증가할 때 매출 % 변화'로 해석합니다.
  - 입력변수도 로그 변환된 경우라면 '탄력성'으로 해석합니다.
- 영향점은 스튜던트 잔차와 레버리지를 함께 반영한 Cook의 거리로 판단합니다.
  - 같은 점포의 영업일수가 크게 줄어든 두 달은 단순 선형 회귀 모형에서는 레버리지가 작았습니다.
  - 영업일수를 함께 넣은 다중 선형 회귀 모형에서는 Cook의 거리가 가장 큰 영향점이었습니다.
- 영향점은 `renovation`(리뉴얼 공사 기간)으로 확인되었습니다.
  - 입력 오류가 아니라 평상시가 아닌 달로 보고 제외 후 모형을 다시 적합했습니다.
- 점포 규모 변수를 모두 넣으면 면적과 품목 수의 분산팽창지수가 10을 넘었습니다.
  - 품목 수를 빼면 분산팽창지수는 10 아래로 내려가지만 주차면수는 면적을 통제하면 유의하지 않아 함께 뺐습니다.
  - 두 변수를 빼도 조정된 결정계수는 그대로였고, 부분 F-검정과 군집 표준오차로 다시 한 검정 모두에서 설명력이 유의하게 줄지 않았습니다.
- 지역은 단독 비교(Welch 분산분석)에서는 유의하지 않았지만 면적을 통제하면 유의했습니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 같은 크기의 점포라면 행사일이 하루 많은 달의 매출이 약 1% 높은 경향이 있습니다.
  - 과거 실적을 비교한 결과이므로 행사를 하루 늘리면 매출이 그만큼 늘어난다는 뜻은 아니며, 행사 효과는 실험이나 행사 전후 비교로 확인합니다.
- 지역 간 매출 차이는 점포 크기를 맞춰 놓고 비교해야 드러납니다.
  - 단순 평균 비교만으로 지역 전략을 세우면 안 됩니다.
- 큰 점포일수록 매출이 크다는 것은 점포끼리 비교한 결과입니다.
  - 회사는 매출이 잘 나올 상권에 큰 점포를 내므로, 이 결과를 '점포를 넓히면 매출이 늘어난다'로 읽으면 안 됩니다.
  - 증축이나 리뉴얼의 효과를 알고 싶다면 같은 점포의 공사 전후 매출을 비슷한 다른 점포와 비교해야 합니다.

## End of Document